# BMIN 5200 — Week 10 in-class exercise
## Inside a language model: tokens, embeddings, attention

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LINK::github-repo/blob/main/exercises/week10.ipynb)

**Time:** ~25 minutes · **Pairs with:** Deep learning & LLMs

### Tasks
- Run real clinical text through a real WordPiece tokenizer and watch drug names and abbreviations come apart
- Implement scaled dot-product attention from the formula, in eight lines of numpy, on a clinical sentence
- Measure with cosine similarity that the same word gets a different vector in two different clinical contexts
- Keep an honest accounting of what a 4-million-parameter model does and does not demonstrate

### Background
Clinical language is exactly the kind of language general-purpose tokenizers were not built for: generic drug names, dose abbreviations, and lab shorthand are rare in web text and get chopped into fragments before the model sees them. That happens upstream of everything else, so no amount of prompting or fine-tuning repairs it. If you plan to put an LLM anywhere near a note, this is the first place to look for silent failure.

Setup. Colab already has `torch`; this installs `transformers` and `tokenizers` and takes about
thirty seconds. The model we download, `google/bert_uncased_L-2_H-128_A-2`, is 17 MB and has two
transformer layers — the "Small Language Models" slide, taken literally, so that everything below
runs on a laptop CPU in seconds.

In [ ]:
%pip install -q transformers tokenizers "torch>=2" --extra-index-url https://download.pytorch.org/whl/cpu
import numpy as np
import pandas as pd
import torch
import transformers
from transformers import AutoModel
from tokenizers import BertWordPieceTokenizer
from huggingface_hub import hf_hub_download

transformers.logging.set_verbosity_error()
transformers.logging.disable_progress_bar()
rng = np.random.default_rng(5200)
pd.set_option("display.width", 140)

TINY_BERT = "google/bert_uncased_L-2_H-128_A-2"


def load_tiny_bert():
    """Fetch the vocabulary and the weights. Returns (None, None) if the network is down."""
    try:
        vocabulary_path = hf_hub_download(TINY_BERT, "vocab.txt")
        wordpiece = BertWordPieceTokenizer(vocabulary_path, lowercase=True)
        model = AutoModel.from_pretrained(TINY_BERT)
        model.eval()
        return wordpiece, model
    except Exception as problem:
        print(f"Could not reach Hugging Face ({type(problem).__name__}). "
              "Part 2 still runs; Parts 1 and 3 will report that they were skipped.")
        return None, None


wordpiece, tiny_bert = load_tiny_bert()
if wordpiece is not None:
    print(f"Vocabulary: {wordpiece.get_vocab_size():,} tokens")
    print(f"Parameters: {sum(p.numel() for p in tiny_bert.parameters()):,}")

We build the tokenizer straight from `vocab.txt` rather than calling `AutoTokenizer`, because the
mechanism is the point. WordPiece has a fixed vocabulary — 30,522 entries here — and any text you
give it must be expressed as a sequence of entries from that list. There is no option to fail.
Whatever is not in the list gets broken into pieces that are, and `##` marks a piece that
continues the previous one.

## Part 1 — Tokenizing a medication list

This vocabulary was learned from Wikipedia and BookCorpus, where `metoprolol` is vanishingly
rare and `hyperlipidemia` barely appears. Watch what survives intact and what does not. This is
the "Representing Text Input" slide, applied to text you would actually find in a chart.

In [ ]:
clinical_notes = [
    "Patient started on metoprolol 25 mg PO BID for rate control.",
    "History of hyperlipidemia and type 2 diabetes mellitus.",
    "Ondansetron 4 mg IV q.i.d. p.r.n. nausea.",
    "SpO2 94% on room air; no acute distress.",
    "Continue levothyroxine; recheck TSH in 6 weeks.",
]

if wordpiece is None:
    print("Skipped: the tokenizer could not be downloaded.")
else:
    for note in clinical_notes:
        pieces = wordpiece.encode(note, add_special_tokens=False).tokens
        print(note)
        print("   " + "  ".join(pieces) + f"    [{len(pieces)} tokens]\n")

`metoprolol` became `met ##op ##rol ##ol`. `hyperlipidemia` became `hyper ##lip ##ide ##mia`.
`q.i.d.` became six tokens, none of which is a dosing frequency. `SpO2` became `sp ##o ##2`.
Worst of all, `ondansetron` starts with the token `on` — the English preposition — so the model's
first impression of an antiemetic is a function word.

The table below quantifies it: how many vocabulary entries each term costs, and whether the term
exists in the vocabulary at all.

In [ ]:
clinical_terms = ["metoprolol", "hyperlipidemia", "levothyroxine", "ondansetron",
                  "atorvastatin", "tachycardia", "echocardiogram", "pneumonia",
                  "diabetes", "aspirin", "insulin"]

if wordpiece is None:
    print("Skipped: the tokenizer could not be downloaded.")
else:
    vocabulary = wordpiece.get_vocab()
    rows = []
    for term in clinical_terms:
        pieces = wordpiece.encode(term, add_special_tokens=False).tokens
        rows.append({"term": term,
                     "tokens": len(pieces),
                     "in vocabulary": term in vocabulary,
                     "pieces": " ".join(pieces)})
    print(pd.DataFrame(rows).to_string(index=False))

Three terms survive whole — `pneumonia`, `diabetes`, `insulin` — because they are common enough
on the open web to have earned a slot. Every other term is fragments, and not one generic drug
name made it in; even `aspirin` costs three tokens. That is not a cosmetic issue: the model has
to reassemble the concept "beta blocker" from four pieces that each carry meanings of their own,
using only context, every single time. Rare terms get less context in training than common ones,
so the problem is worst exactly where clinical stakes are highest.

Note the cost in context window too. Those eleven terms, which a clinician reads as eleven
things, are 36 tokens to the model, and you pay for every one of them.

## Part 2 — Attention from the formula

Attention is $\text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$, and it is not more than
that. For self-attention on one short sentence we let $Q = K = V = E$, the embedding matrix, so
each token's output is a weighted average of all the tokens' vectors, weighted by how similar
they are to it. The hand-built 4-dimensional embeddings below have named axes so you can read the
similarities off directly: `pain` scores high on symptom and on anatomy, `on` is a function word
and scores near zero on everything.

Complete the TODO. The placeholder gives every token an equal weight, which is a plain average —
and a plain average is what attention would be if you deleted the softmax over similarities.

In [ ]:
tokens = ["patient", "denies", "chest", "pain", "on", "exertion"]
axes = ["person", "negation", "anatomy", "symptom"]

embeddings = np.array([
    [2.5, 0.0, 0.0, 0.4],    # patient
    [0.0, 2.5, 0.0, 0.2],    # denies
    [0.0, 0.3, 2.5, 0.8],    # chest
    [0.0, 0.5, 2.2, 2.5],    # pain
    [0.3, 0.0, 0.0, 0.3],    # on
    [0.0, 0.0, 0.5, 2.2],    # exertion
])
print(pd.DataFrame(embeddings, index=tokens, columns=axes).to_string(), "\n")

queries = keys = values = embeddings
d_k = embeddings.shape[1]

scores = queries @ keys.T / np.sqrt(d_k)          # similarity of every token to every token
# TODO: replace the uniform weights below with softmax applied along each row of `scores`:
#     stabilized = scores - scores.max(axis=1, keepdims=True)   # avoid overflow in exp
#     weights = np.exp(stabilized)
#     weights = weights / weights.sum(axis=1, keepdims=True)
weights = np.ones_like(scores) / len(tokens)

attention_output = weights @ values

print("Attention weights — each row sums to 1 and says where that token looked:")
print(pd.DataFrame(weights.round(2), index=tokens, columns=tokens).to_string())
print("\nOutput vectors, one per token:")
print(pd.DataFrame(attention_output.round(2), index=tokens, columns=axes).to_string())

With the uniform placeholder every row of the output is identical — all six tokens end up as the
same average vector, and the sentence has been flattened into a bag of words. That is the failure
mode attention exists to fix, and it is worth staring at for a second before you fix it.

With the softmax in place, read the matrix by row. `chest` puts about half its weight on `pain`;
`exertion` puts more than half its weight on `pain`; `pain` mostly attends to itself and then to
`chest`. And `on`, having a near-zero embedding, spreads its attention almost uniformly, because
it is similar to nothing. That is all "attention" means here: a similarity-weighted average. No
part of it knows that `denies` negates the symptom.

## Predict before you run

Part 3 takes the same word in two clinical sentences and compares the vectors the model produces.

- "the wound shows no purulent **discharge** ." — a fluid
- "the patient is ready for **discharge** to home ." — a disposition

Before the transformer layers, both are the same row of the same lookup table, so their cosine
similarity is exactly 1.000. **Commit to a number:** after two layers of attention, what will the
cosine similarity be? And will two sentences that use `discharge` in the *same* sense come out
closer together than these two do?

## Part 3 — Contextual embeddings

`last_hidden_state` holds one vector per token after both transformer layers have run, so the
vector at the position of `discharge` has absorbed the rest of its sentence. We compare that
against the static input embedding — the raw lookup, identical by construction. Fill in the
cosine similarity; the placeholder returns a bare dot product, which is not bounded to
$[-1, 1]$ and will give you numbers that cannot be similarities.

In [ ]:
def cosine_similarity(first, second):
    """Cosine similarity: the dot product of two vectors after both are scaled to length 1."""
    # TODO: divide by the product of the two vector norms (np.linalg.norm).
    return float(first @ second)


def contextual_vector(sentence, target_word):
    """The model's vector for one word, after it has seen the whole sentence."""
    encoded = wordpiece.encode(sentence)
    position = encoded.tokens.index(target_word)
    with torch.no_grad():
        output = tiny_bert(input_ids=torch.tensor([encoded.ids]))
    return output.last_hidden_state[0, position].numpy(), encoded.ids[position]


fluid_sense = "the wound shows no purulent discharge ."
disposition_sense = "the patient is ready for discharge to home ."
same_sense_as_disposition = "we plan discharge to a rehabilitation facility ."
same_sense_as_fluid = "the incision has serous discharge at the margin ."

if tiny_bert is None:
    print("Skipped: the model could not be downloaded.")
else:
    static_lookup = tiny_bert.embeddings.word_embeddings.weight.detach().numpy()
    fluid, token_id = contextual_vector(fluid_sense, "discharge")
    disposition, _ = contextual_vector(disposition_sense, "discharge")
    disposition_2, _ = contextual_vector(same_sense_as_disposition, "discharge")
    fluid_2, _ = contextual_vector(same_sense_as_fluid, "discharge")

    print(f"static embedding, both sentences   : "
          f"{cosine_similarity(static_lookup[token_id], static_lookup[token_id]):.3f}   "
          f"(the same row of the lookup table twice: this has to come out 1.000)")
    print(f"contextual, fluid vs disposition   : {cosine_similarity(fluid, disposition):.3f}")
    print(f"contextual, disposition vs same    : {cosine_similarity(disposition, disposition_2):.3f}")
    print(f"contextual, fluid vs same          : {cosine_similarity(fluid, fluid_2):.3f}")

With the cosine written correctly: the two senses land at 0.662, while two sentences sharing a
sense land at 0.812 and 0.798. The word went into the model as one vector and came out as two
measurably different ones, sorted by meaning. That single number is the whole content of the
phrase "contextual embeddings", and it is the reason transformers replaced the fixed-vector
approaches on the "Representing Text Input" slide.

Try the same comparison on `positive` — "a positive blood culture" against "a positive attitude
toward treatment" — and you get 0.873. The separation is real but much weaker, which is what a
two-layer model should be expected to manage.

## Scope and limitations

Be precise about what you just watched, because the gap matters. This model has **4,385,920
parameters**. A frontier LLM has on the order of hundreds of billions — roughly five orders of
magnitude more, trained on roughly five orders of magnitude more text.

Nothing in this notebook demonstrates reasoning. You saw a lookup table, a similarity-weighted
average, and a fixed vocabulary. Two transformer layers moved a word vector measurably toward its
sense; they did not decide anything, plan anything, or know that `denies` negates a symptom. Any
claim that a language model "understands" a clinical note has to be argued for on top of these
mechanics, not read off them.

## Discussion

1. `ondansetron` tokenizes to `on ##dan ##set ##ron`, starting with an English preposition. What
   would you need to measure to find out whether this actually degrades a clinical task, rather
   than just looking alarming?
2. Attention gave `denies` no special power over `pain` — it is one vector among six in a
   weighted average. Real models handle negation reasonably well anyway. Where do you think that
   capability lives, if not in the attention formula?
3. Domain tokenizers trained on clinical text keep `metoprolol` whole. That costs you the ability
   to start from a public pretrained model. Which way would you go for a health-system deployment,
   and what evidence would you want before committing?

## Solutions

Completed versions of the TODOs, as markdown so they do not run.

**Part 2 — softmax attention:**

```python
scores = queries @ keys.T / np.sqrt(d_k)
stabilized = scores - scores.max(axis=1, keepdims=True)
weights = np.exp(stabilized)
weights = weights / weights.sum(axis=1, keepdims=True)
attention_output = weights @ values
```

Which gives, rounded to two places:

```
          patient  denies  chest  pain    on  exertion
patient      0.78    0.03   0.04  0.05  0.05      0.05
denies       0.03    0.76   0.05  0.08  0.03      0.04
chest        0.01    0.02   0.38  0.53  0.01      0.05
pain         0.00    0.01   0.12  0.79  0.00      0.07
on           0.20    0.13   0.15  0.19  0.14      0.18
exertion     0.03    0.03   0.09  0.56  0.03      0.26
```

Subtracting the row maximum before exponentiating changes nothing mathematically — softmax is
invariant to a constant shift — and prevents `np.exp` from overflowing on large scores.

**Part 3 — cosine similarity:**

```python
def cosine_similarity(first, second):
    return float(first @ second / (np.linalg.norm(first) * np.linalg.norm(second)))
```

Giving 1.000 for the static embeddings, 0.662 across the two senses, and 0.812 / 0.798 within a
sense.